# Local Agentic AI with Gemma 4 (12B) & Apple MLX — Lab 4: StockAgent — Autonomous Market Screener & Stock Recommender

> ⚠️ **IMPORTANT FINANCIAL & EDUCATIONAL DISCLAIMER:**  
> **AI models (including local LLMs like Gemma 4) can and do make mistakes, hallucinate, or misinterpret real-time market data.**  
> **This notebook and tutorial are strictly for educational and technical demonstration purposes only.**  
> **I am a software engineer, not a certified financial advisor or registered broker-dealer, and I do NOT have financial expertise to recommend stocks.**  
> **Never make trades or investments based solely on AI outputs. Always do your own due diligence (DYOR) and consult a certified financial advisor before risking real capital.**

---

In Lab 1, we established our local **Google Gemma 4 (12B)** inference engine on **Apple MLX**. In Lab 2, we mastered **Pydantic schemas**, live **Serper API** quotes, and temporal date grounding. In Lab 3, we built modular valuation routines to evaluate market leaders and generate executive memos.

Now, in Lab 4, we scale our agentic capabilities to the live open web: building **StockAgent**, a fully autonomous financial intelligence copilot that scrapes live market tables from Yahoo Finance, enforces strict institutional risk boundaries to eliminate penny stocks, and synthesizes the **3 most promising stock opportunities**.

| Feature & Safety Boundary | Traditional Stock Screeners | StockAgent (Local Gemma 4 + Pydantic + Apple MLX) |
|---|---|---|
| **Query Flexibility** | Hardcoded filter menus & dropdowns | Natural language conversational intelligence |
| **Risk Guardrails** | Manual ticker checking | **Automated penny stock purge** ($Price \ge \$5$, $Cap \ge \$1B$) |
| **Live Extraction** | Paid proprietary vendor APIs | Direct, autonomous web scraping from Yahoo Finance (Zero mock data) |
| **Decision Synthesis** | Raw numerical data dumps | Technical memos with growth catalysts, stop-loss & risk evaluation |
| **Data Sovereignty** | Cloud tracking of investor watchlists | **100% Local & Private analysis** on Apple Silicon unified memory |

> **Core Philosophy:** Real-world trading copilots must combine real-time web awareness with ironclad risk guardrails. By combining **Gemma 4** with strongly typed **Pydantic tool schemas**, StockAgent screens live market candidates, purges high-risk penny stocks, and delivers institutional-grade investment briefings with zero cloud subscription fees.

## 1. Prerequisites & Execution Architecture

StockAgent operates through a deterministic, three-phase autonomous execution cycle:
1. **Intent Analysis & Tool Routing**: Gemma 4 evaluates the user's market query against a unified decision contract (`StockAgentDecision`), selecting market scraping (`scrape_market`) or financial education (`none`).
2. **Safety-Sandboxed Live Scraping**: Python fetches live HTML tables from Yahoo Finance (`most_active` or `gainers`) via `requests` and `BeautifulSoup4`, then applies mathematical filters to purge sub-$5 penny stocks and illiquid micro-caps in real time.
3. **Financial Synthesis**: Gemma 4 evaluates the filtered candidates and formats a structured Markdown investment memo recommending the top 3 opportunities with strict temporal grounding.

In [1]:
# ── Library Imports & Client Initialization ──────────────────────────────────
from pydantic import BaseModel, Field
from typing import Optional, Literal, Dict, Any, List
import json, os, re, platform, requests
from datetime import datetime
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import mlx.core as mx
from mlx_vlm import load, generate

# Load Google Gemma 4 (12B) on Apple MLX unified memory
MODEL_PATH = "/Users/abhisheksaxena/AIModels/gemma-4-12B-it-qat-4bit"
print("Loading Gemma 4 (12B) on Apple MLX...")
model, processor = load(MODEL_PATH)

print("✅ Gemma 4 (12B) loaded into unified memory on default GPU device.")
print(f"  Target Model : Gemma 4 (12B-it-qat-4bit)")
print(f"  Host System  : {platform.system()} {platform.machine()} (Apple Silicon)")
print(f"  System Time  : {datetime.now().strftime('%B %d, %Y (%I:%M %p)')}")
print("  Web Scraper  : Requests + BeautifulSoup4 active (Zero Mock Data)")

Loading Gemma 4 (12B) on Apple MLX...
✅ Gemma 4 (12B) loaded into unified memory on default GPU device.
  Target Model : Gemma 4 (12B-it-qat-4bit)
  Host System  : Darwin arm64 (Apple Silicon)
  System Time  : October 04, 2026 (07:29 PM)
  Web Scraper  : Requests + BeautifulSoup4 active (Zero Mock Data)


## 2. Defining Strongly Typed Financial Tool Schemas

To prevent the model from hallucinating invalid parameters or bypassing risk guardrails, we define **two explicit Pydantic schemas**:
1. `YahooScraperParams`: Specifies market section (`most_active`, `gainers`), minimum price boundary ($\ge \$5.00$), and minimum market cap ($\ge \$1.0\text{B}$).
2. `StockFilterParams`: Defines custom filtering criteria including minimum trading volume and screening strategy.

In [2]:
# Tool 1 Schema: Yahoo Finance Market Scraper & Penny Stock Filter
class YahooScraperParams(BaseModel):
    """Parameters for scraping and filtering Yahoo Finance market tables."""
    section: Literal["most_active", "gainers"] = Field(
        "most_active", description="Market section to scrape: 'most_active' or 'gainers'"
    )
    min_price: float = Field(
        5.0, ge=5.0, description="Strict penny stock exclusion: minimum share price in USD (must be >= $5.00)"
    )
    min_market_cap_b: float = Field(
        1.0, ge=0.3, description="Micro-cap exclusion: minimum market capitalization in Billions USD (must be >= $1.0B)"
    )
    limit: int = Field(
        10, ge=3, le=25, description="Maximum number of candidate stocks to return"
    )

# Tool 2 Schema: Fundamental Stock Filter & Screener
class StockFilterParams(BaseModel):
    """Parameters for fine-grained fundamental and technical screening."""
    strategy: Literal["momentum", "value", "growth", "balanced"] = Field(
        "balanced", description="Screening strategy: 'momentum', 'value', 'growth', or 'balanced'"
    )
    max_pe_ratio: Optional[float] = Field(
        None, ge=1.0, description="Optional maximum P/E ratio ceiling for value screening"
    )

print("✓ Financial tool schemas defined successfully.")
print("Sample JSON Schema (YahooScraperParams):")
print(json.dumps(YahooScraperParams.model_json_schema(), indent=2))

✓ Financial tool schemas defined successfully.
Sample JSON Schema (YahooScraperParams):
{
  "description": "Parameters for scraping and filtering Yahoo Finance market tables.",
  "properties": {
    "limit": {
      "default": 10,
      "description": "Maximum number of candidate stocks to return",
      "maximum": 25,
      "minimum": 3,
      "title": "Limit",
      "type": "integer"
    },
    "min_market_cap_b": {
      "default": 1.0,
      "description": "Micro-cap exclusion: minimum market capitalization in Billions USD (must be >= $1.0B)",
      "minimum": 0.3,
      "title": "Min Market Cap B",
      "type": "number"
    },
    "min_price": {
      "default": 5.0,
      "description": "Strict penny stock exclusion: minimum share price in USD (must be >= $5.00)",
      "minimum": 5.0,
      "title": "Min Price",
      "type": "number"
    },
    "section": {
      "default": "most_active",
      "description": "Market section to scrape: 'most_active' or 'gainers'",
      "enum"

## 3. Unified StockAgent Decision Contract

To give our agent structured autonomous reasoning, we define `StockAgentDecision`. In a single inference pass, Gemma 4 evaluates the incoming request, selects the appropriate tool, and constructs verified tool arguments.

In [3]:
class StockAgentDecision(BaseModel):
    """Unified decision and routing contract for StockAgent."""
    tool: Literal["scrape_market", "none"] = Field(..., description="Selected tool: 'scrape_market' or 'none'")
    reasoning: str = Field(..., description="Explicit analytical reasoning justifying tool choice")
    scraper_params: Optional[YahooScraperParams] = Field(None, description="Parameters for Yahoo Finance scraper")
    filter_params: Optional[StockFilterParams] = Field(None, description="Parameters for custom stock screening")

print("✓ Unified StockAgent Decision Schema compiled and validated with Pydantic.")

✓ Unified StockAgent Decision Schema compiled and validated with Pydantic.


## 4. Live Yahoo Finance Scraper, Penny Stock Purge & Temporal Grounding Engine

Our scraper connects live to Yahoo Finance's web tables, dynamically parses numerical strings (converting billions and trillions), and applies strict risk filtering. **Crucially, zero mock data is used**—every price, market cap, and volume is extracted live from the active market, stamped with today's real-time system clock.

In [4]:
# Helper parser functions for numerical values and penny-stock filtration
def parse_num(txt: str) -> float:
    """Parses financial numbers with T, B, M suffixes into standard floats."""
    if not txt or txt in ('-', '--', 'N/A'):
        return 0.0
    txt = txt.replace(',', '').replace('$', '').strip()
    mult = 1.0
    if txt.endswith('T'):
        mult = 1e12
        txt = txt[:-1]
    elif txt.endswith('B'):
        mult = 1e9
        txt = txt[:-1]
    elif txt.endswith('M'):
        mult = 1e6
        txt = txt[:-1]
    elif txt.endswith('%'):
        txt = txt[:-1]
    try:
        return float(txt) * mult
    except ValueError:
        return 0.0

print("✓ Numerical parsers and live penny-stock filtration engine ready.")

✓ Numerical parsers and live penny-stock filtration engine ready.


In [5]:
# Live Yahoo Finance Scraper Tool with Temporal Grounding
def scrape_yahoo_finance(params: YahooScraperParams) -> Dict[str, Any]:
    """Scrapes Yahoo Finance live tables, applies penny-stock filtration, and injects temporal grounding."""
    endpoint_map = {
        "most_active": "https://finance.yahoo.com/markets/stocks/most-active/?start=0&count=25",
        "gainers": "https://finance.yahoo.com/markets/stocks/gainers/?start=0&count=25"
    }
    url = endpoint_map.get(params.section, endpoint_map["most_active"])
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    }

    now_dt = datetime.now()
    live_timestamp = now_dt.strftime("%B %d, %Y (%I:%M %p)")
    market_date = now_dt.strftime("%Y-%m-%d")

    scraped_stocks = []
    purged_count = 0
    
    resp = requests.get(url, headers=headers, timeout=12)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    for row in soup.find_all('tr'):
        ticker_cell = row.find('td', {'data-testid-cell': 'ticker'})
        if not ticker_cell:
            continue
        ticker = ticker_cell.get_text(strip=True)
        name_cell = row.find('td', {'data-testid-cell': 'companyshortname.raw'})
        company = name_cell.get_text(strip=True) if name_cell else ticker
        
        price_cell = row.find('td', {'data-testid-cell': 'intradayprice'})
        price_span = price_cell.find('span', {'data-testid': 'change'}) if price_cell else None
        price = parse_num(price_span.get_text(strip=True) if price_span else '')
        
        pct_cell = row.find('td', {'data-testid-cell': 'percentchange'})
        change_pct = pct_cell.get_text(strip=True) if pct_cell else '0.0%'
        
        vol_cell = row.find('td', {'data-testid-cell': 'dayvolume'})
        volume = vol_cell.get_text(strip=True) if vol_cell else '0.0M'
        
        mcap_cell = row.find('td', {'data-testid-cell': 'intradaymarketcap'})
        market_cap_raw = parse_num(mcap_cell.get_text(strip=True)) if mcap_cell else 0.0
        market_cap_b = round(market_cap_raw / 1e9, 2)
        
        pe_cell = row.find('td', {'data-testid-cell': 'peratio.lasttwelvemonths'})
        pe_val = pe_cell.get_text(strip=True) if pe_cell and pe_cell.get_text(strip=True) not in ('-', '--') else 'N/A'
        
        # Risk Boundary: Purge penny stocks & micro-caps
        if price < params.min_price or market_cap_b < params.min_market_cap_b:
            purged_count += 1
            continue
            
        scraped_stocks.append({
            'symbol': ticker,
            'name': company,
            'price': price,
            'change_pct': change_pct,
            'volume': volume,
            'market_cap_b': market_cap_b,
            'pe_ratio': pe_val
        })

    return {
        "section": params.section,
        "temporal_grounding": {
            "scan_timestamp": live_timestamp,
            "market_date": market_date,
            "source": "Yahoo Finance Live Telemetry"
        },
        "total_scraped": len(scraped_stocks) + purged_count,
        "purged_penny_stocks": purged_count,
        "min_price_threshold": f"${params.min_price:.2f}",
        "min_market_cap_threshold": f"${params.min_market_cap_b:.1f}B",
        "candidates": scraped_stocks[:params.limit]
    }

TOOL_REGISTRY = {
    "scrape_market": scrape_yahoo_finance
}
anchor_init = datetime.now().strftime("%B %d, %Y (%I:%M %p)")
print("✓ Live Yahoo Finance scraper & Penny Stock Purge Engine initialized.")
print(f"✓ Temporal Grounding: ACTIVE [Anchor: {anchor_init}]")

✓ Live Yahoo Finance scraper & Penny Stock Purge Engine initialized.
✓ Temporal Grounding: ACTIVE [Anchor: October 04, 2026 (07:29 PM)]


## 5. Autonomous StockAgent Execution Loop

We assemble the complete three-phase workflow into `run_stock_agent(user_query: str) -> str`:
- **Phase 1: Intent Analysis & Routing**: Gemma 4 evaluates the user prompt and outputs structured decision JSON.
- **Phase 2: Sandboxed Live Scraping**: Dispatches tool to Yahoo Finance, purges penny stocks, and attaches live timestamp.
- **Phase 3: Financial Synthesis**: Gemma 4 crafts an institutional briefing with mandatory **educational disclaimer and due diligence warnings**.

In [6]:
def run_stock_agent(user_query: str) -> str:
    print(f"\n💬 User Query: \"{user_query}\"")
    now_str = datetime.now().strftime("%B %d, %Y (%I:%M %p)")
    
    # Phase 1: Structured Intent Analysis & Routing with Gemma 4
    schema_desc = json.dumps(StockAgentDecision.model_json_schema(), indent=2)
    routing_messages = [
        {
            "role": "system",
            "content": (
                f"You are StockAgent, an autonomous market research copilot. Today's date is {now_str}. "
                "Analyze the user's query and decide which tool to use. Return strictly valid JSON adhering to this schema:\n"
                f"{schema_desc}\n"
                "Rules:\n"
                "1. If user requests scraping, screening, or stock recommendations, tool must be 'scrape_market'.\n"
                "2. If user requests gainers or top performers, set scraper_params.section to 'gainers'. Otherwise 'most_active'.\n"
                "3. If query is purely conceptual or educational without requiring stock data, tool must be 'none'."
            )
        },
        {"role": "user", "content": user_query}
    ]
    routing_prompt = processor.apply_chat_template(routing_messages, tokenize=False, add_generation_prompt=True)
    res_raw = generate(model, processor, prompt=routing_prompt, max_tokens=250, verbose=False)
    res_text = res_raw.text.strip() if hasattr(res_raw, 'text') else str(res_raw).strip()
    
    json_match = re.search(r'\{.*\}', res_text, re.DOTALL)
    if json_match:
        try:
            decision = StockAgentDecision.model_validate_json(json_match.group(0))
        except Exception:
            decision = StockAgentDecision(
                tool="scrape_market" if any(k in user_query.lower() for k in ["gain", "active", "recommend", "stock"]) else "none",
                reasoning="Extracted routing intent from user prompt."
            )
    else:
        decision = StockAgentDecision(
            tool="scrape_market" if any(k in user_query.lower() for k in ["gain", "active", "recommend", "stock"]) else "none",
            reasoning="Default heuristic fallback routing."
        )
        
    print(f"🤖 Agent Reasoning: {decision.reasoning}")
    print(f"🔧 Tool Selected  : {decision.tool}")
    
    if decision.tool == "none":
        conceptual_messages = [
            {
                "role": "system",
                "content": (
                    f"You are StockAgent, an educational financial AI assistant. Today's date is {now_str}. "
                    "Answer the conceptual question thoroughly. Always emphasize that AI can make mistakes, that this is strictly "
                    "for educational purposes only, that you do not have financial expertise to recommend stocks, and that users must do their own due diligence."
                )
            },
            {"role": "user", "content": user_query}
        ]
        prompt = processor.apply_chat_template(conceptual_messages, tokenize=False, add_generation_prompt=True)
        ans = generate(model, processor, prompt=prompt, max_tokens=650, verbose=False)
        return ans.text.strip() if hasattr(ans, 'text') else str(ans).strip()
        
    # Phase 2: Live Tool Execution with Penny Stock Purge & Temporal Grounding
    tool_func = TOOL_REGISTRY.get(decision.tool, scrape_yahoo_finance)
    params = decision.scraper_params or YahooScraperParams(
        section="gainers" if "gainer" in user_query.lower() or "momentum" in user_query.lower() else "most_active",
        min_price=10.0 if "10" in user_query else 5.0,
        min_market_cap_b=2.0 if "2b" in user_query.lower() or "2 billion" in user_query.lower() else 1.0
    )
    market_data = tool_func(params)
    current_time_str = market_data.get("temporal_grounding", {}).get("scan_timestamp") or now_str
    
    print(f"⚙️ Scraper Output : Scraped {market_data['total_scraped']} tickers | Purged {market_data['purged_penny_stocks']} penny/micro-caps")
    print(f"   Safe Candidates: {[s['symbol'] for s in market_data['candidates']]}")
    print(f"🕒 Temporal Anchor: {current_time_str}")
    
    # Phase 3: Financial Synthesis with Educational Disclaimer & Due Diligence Notice
    synthesis_messages = [
        {
            "role": "system",
            "content": (
                f"You are StockAgent, an educational financial analysis copilot running locally on Apple MLX. "
                f"The verified market timestamp is strictly {current_time_str}. "
                "Structure your response with:\n"
                f"1. Header: **Report Date:** {current_time_str} (Live Market Telemetry)\n"
                "2. ⚠️ **EDUCATIONAL DISCLAIMER**: State clearly that AI models make mistakes, this report is for educational and technical demonstration purposes only, you do not possess certified financial expertise to recommend stocks, and users must do their own due diligence before risking real capital.\n"
                "3. **Top 3 Stock Opportunities**: Select the 3 most promising stocks from the provided live candidates. For each include: Ticker & Name, Key Financials (Price, Market Cap, Volume, P/E), Investment Thesis & Catalysts, and Risk Factors / Stop-Loss Level.\n"
                "4. **Executive Summary & Portfolio Risk Note**."
            )
        },
        {
            "role": "user",
            "content": f"User Request: {user_query}\n\nLive Verified Market Data:\n{json.dumps(market_data, indent=2)}\n\nGenerate the complete report."
        }
    ]
    prompt = processor.apply_chat_template(synthesis_messages, tokenize=False, add_generation_prompt=True)
    ans = generate(model, processor, prompt=prompt, max_tokens=850, verbose=False)
    return ans.text.strip() if hasattr(ans, 'text') else str(ans).strip()

print("✓ Autonomous StockAgent execution loop compiled and ready on Apple MLX.")

✓ Autonomous StockAgent execution loop compiled and ready on Apple MLX.


## 6. Live Evaluation — Scenario 1: Most Active Market Screener

We deploy StockAgent to audit today's most heavily traded stocks on Yahoo Finance, strictly enforce our penny stock exclusion boundary ($Price \ge \$5$, $Cap \ge \$1\text{B}$), and recommend the **3 most promising stocks**.

In [7]:
# Scenario 1: Most Active Stocks Screener & Top 3 Recommendations
ans1 = run_stock_agent(
    "Scrape Yahoo Finance for today's most active stocks, filter out all penny stocks under $5 and micro-caps, and recommend the 3 most promising opportunities."
)
display(Markdown(f"### Scenario 1 Recommendations:\n{ans1}"))


💬 User Query: "Scrape Yahoo Finance for today's most active stocks, filter out all penny stocks under $5 and micro-caps, and recommend the 3 most promising opportunities."
🤖 Agent Reasoning: The user wants to identify the most active stocks for today while applying specific filters (price > $5 and market cap > $1B). I will use the 'most_active' section and set the appropriate price and market cap thresholds. After scraping, I will analyze the results to provide the 3 most promising recommendations.
🔧 Tool Selected  : scrape_market
⚙️ Scraper Output : Scraped 25 tickers | Purged 3 penny/micro-caps
   Safe Candidates: ['NKE', 'NVDA', 'SPCX', 'INTC', 'CTVA', 'NU', 'NOK', 'AAL', 'RIVN', 'WBD']
🕒 Temporal Anchor: October 04, 2026 (07:29 PM)


### Scenario 1 Recommendations:
**Report Date:** October 04, 2026 (07:29 PM) (Live Market Telemetry)

⚠️ **EDUCATIONAL DISCLAIMER**: AI models can make mistakes, and this report is for educational and technical demonstration purposes only. I do not possess certified financial expertise to recommend stocks. This analysis is based on the provided telemetry and does not constitute financial advice. Users must perform their own due diligence and consult with a certified financial advisor before risking real capital.

---

### **Top 3 Stock Opportunities**

Based on the filtered live telemetry (excluding penny stocks and micro-caps), the following three candidates show the most compelling technical and fundamental profiles for educational analysis:

#### **1. SpaceX (SPCX)**
*   **Key Financials:**
    *   **Price:** $158.96 (+7.35%)
    *   **Market Cap:** $2.095 Trillion
    *   **Volume:** 119.859M
    *   **P/E Ratio:** N/A (Growth/Pre-profit stage)
*   **Investment Thesis & Catalysts:** SPCX is showing significant bullish momentum today with a 7.35% price jump. As a leader in the aerospace and satellite internet sectors, the primary catalyst is the continued expansion of the Starlink constellation and the increasing frequency of heavy-lift launches. The high volume suggests strong institutional interest.
*   **Risk Factors / Stop-Loss Level:** High volatility due to regulatory hurdles and launch failures. **Stop-Loss Level:** $145.00 (approx. 8% below current price).

#### **2. NVIDIA Corporation (NVDA)**
*   **Key Financials:**
    *   **Price:** $233.95 (+1.34%)
    *   **Market Cap:** $5.649 Trillion
    *   **Volume:** 135.168M
    *   **P/E Ratio:** 29.58
*   **Investment Thesis & Catalysts:** NVDA remains the cornerstone of the AI infrastructure trade. Despite its massive market cap, it maintains a relatively moderate P/E ratio for a high-growth tech giant. The catalyst is the sustained demand for H100/B200 series chips and the integration of AI into enterprise software.
*   **Risk Factors / Stop-Loss Level:** Concentration risk and potential for "AI fatigue" in the market. **Stop-Loss Level:** $215.00.

#### **3. Nu Holdings Ltd. (NU)**
*   **Key Financials:**
    *   **Price:** $13.43 (+1.05%)
    *   **Market Cap:** $64.88 Billion
    *   **Volume:** 78.03M
    *   **P/E Ratio:** 18.30
*   **Investment Thesis & Catalysts:** NU presents a compelling "Growth at a Reasonable Price" (GARP) profile. With a P/E of 18.30, it is significantly more undervalued than many of its peers in the fintech space. The catalyst is the continued expansion of its digital banking platform across Latin America, capturing high-margin deposits.
*   **Risk Factors / Stop-Loss Level:** Currency fluctuations and regional economic instability. **Stop-Loss Level:** $11.80.

---

### **Executive Summary & Portfolio Risk Note**


## 7. Live Evaluation — Scenario 2: High-Momentum Market Gainers

Next, we command StockAgent to scan today's top percentage gainers on Yahoo Finance with stricter filters ($Price \ge \$10$ and $Cap \ge \$2\text{B}$) to identify institutional-grade momentum breakout candidates.

In [8]:
# Scenario 2: High-Momentum Market Gainers
ans2 = run_stock_agent(
    "Scan today's top market gainers with a minimum share price of $10 and market cap over $2B, and identify the 3 best momentum plays."
)
display(Markdown(f"### Scenario 2 Recommendations:\n{ans2}"))


💬 User Query: "Scan today's top market gainers with a minimum share price of $10 and market cap over $2B, and identify the 3 best momentum plays."
🤖 Agent Reasoning: The user requested a scan of top market gainers with specific price and market cap criteria, necessitating data retrieval from a live market data source. This aligns with the tool selection rules.
🔧 Tool Selected  : scrape_market
⚙️ Scraper Output : Scraped 25 tickers | Purged 4 penny/micro-caps
   Safe Candidates: ['SPCX', 'NOK', 'NVDA', 'NU', 'INTC', 'AAPL', 'MSFT', 'AMZN']
🕒 Temporal Anchor: October 04, 2026 (07:31 PM)


### Scenario 2 Recommendations:
**Report Date:** October 04, 2026 (07:31 PM) (Live Market Telemetry)

⚠️ **EDUCATIONAL DISCLAIMER**: I am an AI, not a financial advisor. AI can make mistakes, and the data provided is for educational purposes only. I do not have the financial expertise to recommend specific stocks or provide investment advice. You must perform your own due diligence and consult with a certified financial professional before making any trades.

### **Top 3 Momentum Opportunities**

Based on the live telemetry provided, the following candidates meet the criteria (Price > $10, Market Cap > $2B) and show the strongest momentum profiles:

*   **Space Exploration Technologies Corp. (SPCX)**
    *   **Price:** $158.96 (+7.35%)
    *   **Market Cap:** $2.095T
    *   **Financials:** High volume (119.86M) indicates strong institutional interest.
    *   **Catalyst:** Significant price action suggests a breakout or positive news cycle regarding aerospace milestones.
    *   **Suggested Stop-Loss:** $151.00 (approx. 5% below current price).

*   **Nokia Oyj (NOK)**
    *   **Price:** $10.60 (+2.22%)
    *   **Market Cap:** $59.35B
    *   **Financials:** P/E Ratio of 79.75; high volume (75.82M) relative to price.
    *   **Catalyst:** Steady upward momentum in the telecommunications infrastructure sector.
    *   **Suggested Stop-Loss:** $10.15.

*   **NVIDIA Corporation (NVDA)**
    *   **Price:** $233.95 (+1.34%)
    *   **Market Cap:** $5.649T
    *   **Financials:** P/E Ratio of 29.58; massive liquidity with 135.17M volume.
    *   **Catalyst:** Sustained dominance in the AI hardware space; low volatility compared to SPCX but high stability.
    *   **Suggested Stop-Loss:** $225.00.

**Executive Summary & Risk Note**
The current market shows a strong preference for high-cap technology and aerospace. **SPCX** is the primary momentum leader today, while **NVDA** remains the cornerstone of the tech sector. **Risk Note:** High-momentum stocks can experience rapid reversals. Ensure you monitor volume trends and stay aware of macro-economic shifts that could impact these large-cap positions.


## 8. Live Evaluation — Scenario 3: Institutional Risk & Penny Stock Audit

Finally, we test StockAgent's conceptual reasoning by asking it to explain why algorithmic trading agents and institutions strictly filter out penny stocks.

In [9]:
# Scenario 3: Conceptual Penny Stock Risk Analysis
ans3 = run_stock_agent(
    "Explain what qualifies as a penny stock, and why should algorithmic trading agents filter them out before executing recommendations?"
)
display(Markdown(f"### Scenario 3 Analysis:\n{ans3}"))


💬 User Query: "Explain what qualifies as a penny stock, and why should algorithmic trading agents filter them out before executing recommendations?"
🤖 Agent Reasoning: The user's query is conceptual and educational, focusing on financial market definitions and algorithmic trading safety principles. No live stock scraping is required.
🔧 Tool Selected  : none


### Scenario 3 Analysis:
Hello! I am StockAgent, your educational financial assistant. I’m happy to help you break down the mechanics of penny stocks and the logic behind why algorithmic trading systems often treat them with extreme caution.

***

### What Qualifies as a Penny Stock?

While there is no single, universal legal definition, the term "penny stock" generally refers to shares of small companies that trade at very low prices—typically **under $5.00 per share**. However, price is only one part of the equation. To truly qualify as a penny stock in a financial context, a security usually exhibits the following characteristics:

1.  **Low Market Capitalization:** These are often "micro-cap" or "nano-cap" companies. They have a small total market value, meaning they lack the massive infrastructure and resources of blue-chip corporations.
2.  **Low Liquidity:** This is a critical factor. Liquidity refers to how easily an asset can be bought or sold without affecting its price. Penny stocks often have very low trading volume, meaning there may not be enough buyers or sellers at any given moment.
3.  **Limited Public Information:** Unlike large companies that are required to file extensive, audited reports with the SEC (or equivalent regulators), many penny stock issuers provide minimal financial disclosures. This makes it difficult for investors to assess the company's true health.
4.  **Over-the-Counter (OTC) Trading:** While some penny stocks trade on major exchanges (like the NASDAQ), many trade on "Pink Sheets" or OTC markets, which have less stringent listing requirements.

### Why Should Algorithmic Trading Agents Filter Them Out?

Algorithmic trading relies on data, patterns, and mathematical models to execute trades. Penny stocks introduce "noise" and risks that can break these models. Here is why an algorithm would typically filter them out:

#### 1. Extreme Volatility and "Pump and Dump" Schemes
Penny stocks are highly susceptible to manipulation. Because the share price is so low, a relatively small amount of buying pressure can cause the price to skyrocket (a "pump"), followed by a massive sell-off (the "dump"). Algorithms designed for steady growth or mean reversion can be "tricked" by these artificial price spikes, leading to catastrophic losses.

#### 2. The Bid-Ask Spread and Slippage
In liquid markets, the difference between the "bid" (what a buyer offers) and the "ask" (what a seller wants) is tiny. In penny stocks, this spread can be massive. 
*   **Example:** If a stock is priced at $0.10, but the spread is $0.02, you are essentially losing 20% of your value the moment you enter the trade. 
An algorithm trying to execute a large order in a low-liquidity stock might experience **slippage**, where the price moves against the trade so quickly that the execution price becomes much worse than the intended price.

#### 3. Data Integrity Issues
Algorithms are only as good as the data they consume. Because many penny stocks have irregular trading hours, frequent halts


## 9. Series Summary & Masterclass Capstone Review

In this capstone lab, we successfully scaled our local agent to the live open web on Apple Silicon:
- Scraped real-time market data directly from **Yahoo Finance** with **zero mock data**.
- Enforced strict **Pydantic risk boundaries** that purge volatile penny stocks ($Price \ge \$5.00$).
- Delivered institutional-grade **Top 3 stock recommendations** with technical catalysts, stop-loss limits, and mandatory **due diligence disclaimers**.
- Anchored all model inferences to **real-time system timestamps**, completely eliminating date hallucinations.

In [10]:
# Lab 4 & Masterclass Summary Banner
print("=" * 70)
print("  LOCAL AGENTIC AI WITH GEMMA 4 (12B) & APPLE MLX — LAB 4 COMPLETE")
print("=" * 70)
print("✓ Live Yahoo Finance market scraping operational (Zero Mock Data)")
print("✓ Penny stock filter active ($Price >= $5.00, $MarketCap >= $1.0B)")
print("✓ Top 3 promising stock recommendation engine validated")
print("✓ Educational disclaimer & due diligence warnings integrated")
print("✓ Temporal Grounding: Active & Anchored to Real-Time Clock")
print("=" * 70)

  LOCAL AGENTIC AI WITH GEMMA 4 (12B) & APPLE MLX — LAB 4 COMPLETE
✓ Live Yahoo Finance market scraping operational (Zero Mock Data)
✓ Penny stock filter active ($Price >= $5.00, $MarketCap >= $1.0B)
✓ Top 3 promising stock recommendation engine validated
✓ Educational disclaimer & due diligence warnings integrated
✓ Temporal Grounding: Active & Anchored to Real-Time Clock
